# Monitor de Call Credit Spreads
Le posicoes abertas diretamente do MT5, reconstroi as travas automaticamente e calcula P&L de desmontagem.

- Mercado aberto  -> ask para recomprar vendida, bid para vender comprada
- Mercado fechado -> last trade (tick.last ou candle D-1) em ambas as pernas

In [27]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from IPython.display import display

if not mt5.initialize():
    raise RuntimeError(f'Falha ao inicializar MT5: {mt5.last_error()}')
print('MT5 conectado')
print(mt5.account_info())

MT5 conectado
AccountInfo(login=18901295, trade_mode=2, leverage=1, limit_orders=0, margin_so_mode=1, trade_allowed=True, trade_expert=True, margin_mode=2, currency_digits=2, fifo_close=False, balance=8937.0, credit=0.0, profit=-587.0, equity=8350.0, margin=4988.0, margin_free=3362.0, margin_level=167.401764234162, margin_so_call=0.0, margin_so_so=0.0, margin_initial=0.0, margin_maintenance=0.0, assets=0.0, liabilities=0.0, commission_blocked=0.0, name='RAFAEL LANZA CARIOCA', server='XPMT5-PRD', currency='BRL', company='XP Investimentos CCTVM S/A')


In [28]:
# ==============================================================================
# HELPERS
# ==============================================================================

def mercado_aberto():
    """
    Retorna True se o mercado brasileiro estiver aberto agora.
    B3: seg-sex, 10h00-17h55 (horario de Brasilia, horario local da maquina).
    """
    agora = datetime.now()
    if agora.weekday() >= 5:  # sabado=5, domingo=6
        return False
    minutos = agora.hour * 60 + agora.minute
    return (10 * 60) <= minutos <= (17 * 60 + 55)


MERCADO_ABERTO = mercado_aberto()
print(f'Mercado: {"ABERTO" if MERCADO_ABERTO else "FECHADO"}')


def classify_type(info):
    if info is None or info.path is None:
        return 'UNKNOWN'
    path = info.path.upper()
    if 'OPCOES' in path:
        if info.option_right == 0:
            return 'CALL'
        elif info.option_right == 1:
            return 'PUT'
        return 'OPTION'
    return 'SPOT'


def d1_close(ticker):
    """Fechamento do ultimo candle diario."""
    rates = mt5.copy_rates_range(
        ticker,
        mt5.TIMEFRAME_D1,
        datetime.utcnow() - timedelta(days=15),
        datetime.utcnow(),
    )
    if rates is not None and len(rates) >= 2:
        return float(rates[-1]['close'])
    return np.nan


def get_last_trade(ticker):
    """
    Ultimo preco negociado.
    Prioridade: tick.last -> candle D-1 close.
    """
    mt5.symbol_select(ticker, True)
    tick = mt5.symbol_info_tick(ticker)
    if tick and tick.last > 0:
        return float(tick.last), 'tick_last'
    return d1_close(ticker), 'd1_close'


def get_ask(ticker):
    """
    Preco para COMPRAR o ticker (recomprar a perna vendida na desmontagem).
    - Mercado aberto  -> ask
    - Mercado fechado -> last trade
    """
    if not MERCADO_ABERTO:
        return get_last_trade(ticker)
    mt5.symbol_select(ticker, True)
    tick = mt5.symbol_info_tick(ticker)
    if tick and tick.ask > 0:
        return float(tick.ask), 'tick_ask'
    info = mt5.symbol_info(ticker)
    if info and info.ask > 0:
        return float(info.ask), 'info_ask'
    return get_last_trade(ticker)


def get_bid(ticker):
    """
    Preco para VENDER o ticker (vender a perna comprada na desmontagem).
    - Mercado aberto  -> bid
    - Mercado fechado -> last trade
    """
    if not MERCADO_ABERTO:
        return get_last_trade(ticker)
    mt5.symbol_select(ticker, True)
    tick = mt5.symbol_info_tick(ticker)
    if tick and tick.bid > 0:
        return float(tick.bid), 'tick_bid'
    info = mt5.symbol_info(ticker)
    if info and info.bid > 0:
        return float(info.bid), 'info_bid'
    return get_last_trade(ticker)


def get_last_price(ticker):
    """Preco de referencia generico (mid se aberto, last trade se fechado)."""
    if not MERCADO_ABERTO:
        return get_last_trade(ticker)
    bid, fonte_bid = get_bid(ticker)
    ask, fonte_ask = get_ask(ticker)
    if not np.isnan(bid) and not np.isnan(ask) and ask > 0:
        return (bid + ask) / 2, 'mid'
    if not np.isnan(bid):
        return bid, fonte_bid
    return ask, fonte_ask


def get_expiration(ticker):
    mt5.symbol_select(ticker, True)
    info = mt5.symbol_info(ticker)
    if info and info.expiration_time and info.expiration_time > 0:
        try:
            return datetime.fromtimestamp(info.expiration_time).date()
        except Exception:
            pass
    return None


# Fallback para underlyings sem ON/PN na description (ex: VALE)
UNDERLYING_FALLBACK = {
    'VALE': 'VALE3',
    'BOVA': 'BOVA11',
    'SMAL': 'SMAL11',
    'IVVB': 'IVVB11',
}


def infer_underlying(ticker, description):
    base = ticker[:4]
    if isinstance(description, str):
        d = description.upper()
        if re.search(r'\bPN\b', d):
            return f'{base}4'
        if re.search(r'\bON\b', d):
            return f'{base}3'
    if base in UNDERLYING_FALLBACK:
        return UNDERLYING_FALLBACK[base]
    return f'{base}3'


print('Helpers carregados')

Mercado: FECHADO
Helpers carregados


In [29]:
# ==============================================================================
# LE POSICOES ABERTAS DO MT5 E FILTRA CALLS
# ==============================================================================

positions = mt5.positions_get()

if not positions:
    print('Nenhuma posicao aberta encontrada no MT5.')
else:
    print(f'{len(positions)} posicao(oes) encontrada(s) no MT5')

calls_abertas = []

for pos in (positions or []):
    ticker = pos.symbol
    mt5.symbol_select(ticker, True)
    info = mt5.symbol_info(ticker)

    if classify_type(info) != 'CALL':
        continue

    strike = float(info.option_strike) if info and info.option_strike > 0 else np.nan
    expiration = get_expiration(ticker)

    # ignora vencimentos passados (lixo que o MT5 deixa nas posicoes abertas)
    if expiration and expiration < datetime.utcnow().date():
        continue

    underlying = infer_underlying(ticker, info.description if info else None)
    is_short = (pos.type == mt5.POSITION_TYPE_SELL)

    calls_abertas.append({
        'ticker'     : ticker,
        'underlying' : underlying,
        'strike'     : strike,
        'expiration' : expiration,
        'perna'      : 'SELL' if is_short else 'BUY',
        'volume'     : int(pos.volume),
        'entry_price': float(pos.price_open),
    })

df_calls = pd.DataFrame(calls_abertas)
print(f'\n{len(df_calls)} perna(s) de CALL aberta(s):')
display(df_calls)

7 posicao(oes) encontrada(s) no MT5

6 perna(s) de CALL aberta(s):


,ticker,underlying,strike,expiration,perna,volume,entry_price
0,VALED779,VALE3,77.90,2026-04-17,SELL,1000,2.19
1,VALED789,VALE3,78.90,2026-04-17,BUY,1000,1.82
2,WEGED483,WEGE3,47.03,2026-04-17,SELL,200,1.31
3,WEGED493,WEGE3,48.03,2026-04-17,BUY,200,0.98
4,BBASD241,BBAS3,23.77,2026-04-17,SELL,500,0.68
5,BBASD251,BBAS3,24.77,2026-04-17,BUY,500,0.36


In [30]:
# ==============================================================================
# RECONSTROI AS TRAVAS
# Regra: dentro de cada (underlying, expiration, volume),
#   menor strike -> perna SELL  |  maior strike -> perna BUY
# ==============================================================================

LOT_SIZE = 1  # volume no MT5 ja vem em quantidade de acoes (nao em contratos)

travas = []

for (underlying, expiration, volume), g in df_calls.groupby(
    ['underlying', 'expiration', 'volume']
):
    sells = g[g['perna'] == 'SELL'].sort_values('strike')
    buys  = g[g['perna'] == 'BUY' ].sort_values('strike')

    for (_, sell_row), (_, buy_row) in zip(sells.iterrows(), buys.iterrows()):
        if sell_row['strike'] >= buy_row['strike']:
            print(f'Par ignorado: {sell_row["ticker"]} / {buy_row["ticker"]}')
            continue
        travas.append({
            'underlying'  : underlying,
            'expiration'  : expiration,
            'qty'         : volume,
            'sell_ticker' : sell_row['ticker'],
            'sell_strike' : sell_row['strike'],
            'sell_entry'  : sell_row['entry_price'],
            'buy_ticker'  : buy_row['ticker'],
            'buy_strike'  : buy_row['strike'],
            'buy_entry'   : buy_row['entry_price'],
        })

df_travas = pd.DataFrame(travas)
print(f'\n{len(df_travas)} trava(s) reconstruida(s):')
display(df_travas)


3 trava(s) reconstruida(s):


,underlying,expiration,qty,sell_ticker,sell_strike,sell_entry,buy_ticker,buy_strike,buy_entry
0,BBAS3,2026-04-17,500,BBASD241,23.77,0.68,BBASD251,24.77,0.36
1,VALE3,2026-04-17,1000,VALED779,77.90,2.19,VALED789,78.90,1.82
2,WEGE3,2026-04-17,200,WEGED483,47.03,1.31,WEGED493,48.03,0.98


In [31]:
# ==============================================================================
# ENRIQUECIMENTO: PRECOS ATUAIS + METRICAS
# ==============================================================================

fonte_label = 'ask/bid' if MERCADO_ABERTO else 'last_trade'
rows = []

for _, t in df_travas.iterrows():

    spread_width   = t['buy_strike'] - t['sell_strike']
    credit_in      = t['sell_entry'] - t['buy_entry']
    # risco maximo = diferenca dos strikes menos credito recebido
    # voce so opera a diferenca das opcoes, nunca o nocional do ativo
    max_loss_share = spread_width - credit_in
    credit_total   = credit_in     * t['qty'] * LOT_SIZE
    max_loss_total = max_loss_share * t['qty'] * LOT_SIZE

    # mercado aberto  -> ask recompra vendida, bid vende comprada
    # mercado fechado -> last trade nas duas pernas
    sell_now, sell_fonte = get_ask(t['sell_ticker'])
    buy_now,  buy_fonte  = get_bid(t['buy_ticker'])
    spot_now, _          = get_last_price(t['underlying'])

    cost_to_close_share = sell_now - buy_now
    cost_to_close_total = cost_to_close_share * t['qty'] * LOT_SIZE

    pnl_total         = credit_total - cost_to_close_total
    pnl_pct_risco     = (pnl_total / max_loss_total * 100) if max_loss_total else np.nan
    credit_retido_pct = (pnl_total / credit_total   * 100) if credit_total   else np.nan

    dias_para_vencer = (
        (t['expiration'] - datetime.utcnow().date()).days
        if t['expiration'] else np.nan
    )

    if pd.isna(spot_now):
        status = 'SEM_DADOS'
    elif spot_now < t['sell_strike']:
        otm_dist = (t['sell_strike'] - spot_now) / spot_now * 100
        status = f'OTM ({otm_dist:+.1f}%)'
    elif spot_now < t['buy_strike']:
        status = 'ATM'
    else:
        itm_dist = (spot_now - t['sell_strike']) / t['sell_strike'] * 100
        status = f'ITM ({itm_dist:+.1f}%)'

    rows.append({
        'sell_ticker'      : t['sell_ticker'],
        'buy_ticker'       : t['buy_ticker'],
        'underlying'       : t['underlying'],
        'expiration'       : t['expiration'],
        'dias_para_vencer' : dias_para_vencer,
        'sell_strike'      : t['sell_strike'],
        'buy_strike'       : t['buy_strike'],
        'spread_width'     : spread_width,
        'qty_contratos'    : t['qty'],
        'sell_entry'       : t['sell_entry'],
        'buy_entry'        : t['buy_entry'],
        'credit_in'        : credit_in,
        'credit_total'     : credit_total,
        'max_loss_total'   : max_loss_total,
        'spot_now'         : spot_now,
        'sell_now'         : sell_now,
        'buy_now'          : buy_now,
        'sell_fonte'       : sell_fonte,
        'buy_fonte'        : buy_fonte,
        'cost_to_close'    : cost_to_close_total,
        'pnl_total'        : pnl_total,
        'pnl_pct_risco'    : pnl_pct_risco,
        'credit_retido_pct': credit_retido_pct,
        'status'           : status,
    })

df_monitor = pd.DataFrame(rows)
print(f'{len(df_monitor)} trava(s) enriquecida(s)  |  preco usado: {fonte_label}')

3 trava(s) enriquecida(s)  |  preco usado: last_trade


In [32]:
# ==============================================================================
# RELATORIO DETALHADO
# ==============================================================================

pd.set_option('display.float_format', '{:,.2f}'.format)

modo = 'MERCADO ABERTO  |  precos: ask/bid' if MERCADO_ABERTO else 'MERCADO FECHADO  |  precos: last trade'

print('\n' + '=' * 70)
print('  MONITOR DE TRAVAS   --   ' + datetime.now().strftime('%d/%m/%Y %H:%M'))
print(f'  {modo}')
print('=' * 70)

for _, r in df_monitor.iterrows():
    dias = int(r['dias_para_vencer']) if not pd.isna(r['dias_para_vencer']) else '?'
    print(f"""
+- {r['sell_ticker']} / {r['buy_ticker']}  ({r['underlying']})
|  Vence: {r['expiration']}  ({dias} dias)
|  Strikes: {r['sell_strike']:.2f} / {r['buy_strike']:.2f}   Width: {r['spread_width']:.2f}   Qtd: {r['qty_contratos']} contrato(s)
|
|  -- ENTRADA ----------------------------------------------------------------
|  Venda: {r['sell_entry']:.2f}   Compra: {r['buy_entry']:.2f}
|  Credito liq.: {r['credit_in']:.2f}/acao   Total recebido: R$ {r['credit_total']:,.2f}
|  Risco max.:   R$ {r['max_loss_total']:,.2f}
|
|  -- MERCADO ATUAL ({fonte_label}) ---------------------------------------------
|  Spot {r['underlying']}: {r['spot_now']:.2f}   Status: {r['status']}
|  Sell {r['sell_ticker']}: {r['sell_now']:.2f} [{r['sell_fonte']}]   Buy {r['buy_ticker']}: {r['buy_now']:.2f} [{r['buy_fonte']}]
|
|  -- DESMONTAGEM ------------------------------------------------------------
|  Custo p/ fechar: R$ {r['cost_to_close']:,.2f}
|  P&L se desmontar agora: R$ {r['pnl_total']:+,.2f}  ({r['pnl_pct_risco']:+.1f}% do risco)
|  Credito retido: {r['credit_retido_pct']:.1f}%
+----------------------------------------------------------------------------
""")

total_recebido   = df_monitor['credit_total'].sum()
total_risco      = df_monitor['max_loss_total'].sum()
total_pnl        = df_monitor['pnl_total'].sum()
total_close_cost = df_monitor['cost_to_close'].sum()

print('=' * 70)
print('  CONSOLIDADO')
print(f'  Credito total recebido : R$ {total_recebido:>10,.2f}')
print(f'  Risco max. total       : R$ {total_risco:>10,.2f}')
print(f'  Custo p/ fechar tudo   : R$ {total_close_cost:>10,.2f}')
print(f'  P&L liq. se fechar tudo: R$ {total_pnl:>+10,.2f}')
print('=' * 70)


  MONITOR DE TRAVAS   --   05/04/2026 14:30
  MERCADO FECHADO  |  precos: last trade

+- BBASD241 / BBASD251  (BBAS3)
|  Vence: 2026-04-17  (12 dias)
|  Strikes: 23.77 / 24.77   Width: 1.00   Qtd: 500 contrato(s)
|
|  -- ENTRADA ----------------------------------------------------------------
|  Venda: 0.68   Compra: 0.36
|  Credito liq.: 0.32/acao   Total recebido: R$ 160.00
|  Risco max.:   R$ 340.00
|
|  -- MERCADO ATUAL (last_trade) ---------------------------------------------
|  Spot BBAS3: 23.43   Status: OTM (+1.5%)
|  Sell BBASD241: 0.55 [tick_last]   Buy BBASD251: 0.22 [tick_last]
|
|  -- DESMONTAGEM ------------------------------------------------------------
|  Custo p/ fechar: R$ 165.00
|  P&L se desmontar agora: R$ -5.00  (-1.5% do risco)
|  Credito retido: -3.1%
+----------------------------------------------------------------------------


+- VALED779 / VALED789  (VALE3)
|  Vence: 2026-04-17  (12 dias)
|  Strikes: 77.90 / 78.90   Width: 1.00   Qtd: 1000 contrato(s)
|
|

In [33]:
# ==============================================================================
# TABELA RESUMIDA
# ==============================================================================

cols = [
    'sell_ticker', 'buy_ticker', 'underlying',
    'expiration', 'dias_para_vencer',
    'sell_strike', 'buy_strike',
    'spot_now',
    'credit_total', 'cost_to_close',
    'pnl_total', 'pnl_pct_risco', 'credit_retido_pct',
    'status',
]

display(
    df_monitor[cols].rename(columns={
        'dias_para_vencer'  : 'dias_vto',
        'credit_total'      : 'cred_R$',
        'cost_to_close'     : 'custo_fech_R$',
        'pnl_total'         : 'pnl_R$',
        'pnl_pct_risco'     : 'pnl_%_risco',
        'credit_retido_pct' : 'cred_retido_%',
    })
)

,sell_ticker,buy_ticker,underlying,expiration,dias_vto,sell_strike,buy_strike,spot_now,cred_R$,custo_fech_R$,pnl_R$,pnl_%_risco,cred_retido_%,status
0,BBASD241,BBASD251,BBAS3,2026-04-17,12,23.77,24.77,23.43,160.00,165.00,-5.00,-1.47,-3.12,OTM (+1.5%)
1,VALED779,VALED789,VALE3,2026-04-17,12,77.90,78.90,83.69,370.00,880.00,-510.00,-80.95,-137.84,ITM (+7.4%)
2,WEGED483,WEGED493,WEGE3,2026-04-17,12,47.03,48.03,50.25,66.00,138.00,-72.00,-53.73,-109.09,ITM (+6.8%)


In [34]:
# ==============================================================================
# ALERTAS AUTOMATICOS
# ==============================================================================

ALERTA_DIAS_VENCER   = 7    # avisa quando faltam <= N dias
ALERTA_CRED_RETIDO   = 80   # avisa quando capturou >= N% do credito
ALERTA_PERDA_MAX_PCT = 50   # avisa quando perda latente >= N% do risco

alertas = []

for _, r in df_monitor.iterrows():
    nome = f"{r['sell_ticker']}/{r['buy_ticker']}"
    status = str(r['status'])

    # linha de status para TODAS as travas
    if 'ITM' in status:
        itm_pct = (r['spot_now'] - r['sell_strike']) / r['sell_strike'] * 100
        linha = (
            f"[{nome}]  Spot {r['spot_now']:.2f} -- {itm_pct:+.1f}% ITM"
            f"  |  P&L: R$ {r['pnl_total']:+,.0f}  ({r['pnl_pct_risco']:+.1f}% do risco)"
        )
    elif 'ATM' in status:
        linha = (
            f"[{nome}]  Spot {r['spot_now']:.2f} -- ATM"
            f"  |  P&L: R$ {r['pnl_total']:+,.0f}  ({r['pnl_pct_risco']:+.1f}% do risco)"
        )
    else:
        otm_pct = (r['sell_strike'] - r['spot_now']) / r['spot_now'] * 100
        linha = (
            f"[{nome}]  Spot {r['spot_now']:.2f} -- OTM ({otm_pct:+.1f}%)"
            f"  |  P&L: R$ {r['pnl_total']:+,.0f}  ({r['pnl_pct_risco']:+.1f}% do risco)"
        )
    alertas.append(linha)

    # alertas extras abaixo da linha de status
    if not pd.isna(r['dias_para_vencer']) and r['dias_para_vencer'] <= ALERTA_DIAS_VENCER:
        alertas.append(f"  >> Vence em {int(r['dias_para_vencer'])} dias -- considere encerrar.")

    if not pd.isna(r['credit_retido_pct']) and r['credit_retido_pct'] >= ALERTA_CRED_RETIDO:
        alertas.append(f"  >> {r['credit_retido_pct']:.0f}% do credito retido -- pode valer fechar.")

print('\n' + '=' * 60)
print('  ALERTAS')
print('=' * 60)
if alertas:
    for a in alertas:
        print(' ', a)
else:
    print('  Nenhum alerta -- todas as travas dentro dos parametros.')
print('=' * 60)


  ALERTAS
  [BBASD241/BBASD251]  Spot 23.43 -- OTM (+1.5%)  |  P&L: R$ -5  (-1.5% do risco)
  [VALED779/VALED789]  Spot 83.69 -- +7.4% ITM  |  P&L: R$ -510  (-81.0% do risco)
  [WEGED483/WEGED493]  Spot 50.25 -- +6.8% ITM  |  P&L: R$ -72  (-53.7% do risco)


In [35]:
mt5.shutdown()
print('MT5 desconectado')

MT5 desconectado
